# HEAT3: transfer entropy between companies on 25+ years of daily data

**Question:** does one S&P 500 stock's recent return tell us about another stock's return over the next
**1, 2, 5 or 10 trading days**, beyond what that stock's own past already tells us, positively or negatively,
and is it an edge after 2 bps?

**Data:** daily adjusted closes from **Yahoo Finance, 2000 onward**, downloaded once and kept on Drive.
Caveat for the write-up: Yahoo has no prices for companies that left the index, so the universe is today's
S&P 500 traced back (survivorship bias).

**How:**
1. **Transfer entropy between every pair of stocks**, every month, from the previous three years only:
   stock A's last day → stock B's next 1 / 2 / 5 / 10 days, after removing what B's own last day explains.
2. **Each stock's 5 peers = its 5 highest-TE sources, positive or negative.**
3. **TE becomes the models' feature:** the last 20 days of the stock and its peers, each peer's signed TE
   strength, and the TE-weighted peer signal go into BiLSTM, CNN-ResNet and Transformer models.
4. **Controls**, same model and budget: no peers, 5 random peers, 5 most *correlated* stocks. An **edge** must
   beat all three, earn more than 2 bps net on the test days (2019 onward; intervals resample 21-day blocks),
   hold in both halves, and survive false-discovery control over 12 cells.

The rule is registered in
[`configs/heat_v3.yaml`](https://github.com/Gaire-commits/Transfer-entropy-/blob/main/configs/heat_v3.yaml)
(2026-11-23), before any daily result. The 5-minute experiments (heat_v1, heat_v2) are still here: set the
environment variable `HEAT_CONFIG` before Setup to run one of them.

**The order:** pick the GPU → Setup → check run (inside the training years: train to 2010, score 2013–2014)
→ registered run → result → release days. Section 4 (tuning) is done: the default settings are used.

**Before anything else, pick the GPU:** Runtime → Change runtime type → A100, L4 or T4 GPU. Changing the
runtime type later resets the session, and then Setup must be run again. Don't use "Run all" once you reach
section 5: it scores the test days.

## 1. Setup

In [2]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.ismount("/content/drive"):
        raise SystemExit("Google Drive did not mount. Runtime -> Restart session, then run Setup again.")
    REPO = "/content/Transfer-entropy-"
    if not os.path.exists(REPO):
        !git clone -q https://github.com/Gaire-commits/Transfer-entropy-.git {REPO}
    else:
        !git -C {REPO} pull -q
    os.chdir(REPO)
    !pip install -q pyyaml yfinance
else:
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(REPO)
sys.path.insert(0, REPO)
for name in [m for m in sys.modules if m == "heat" or m.startswith("heat.")]:
    del sys.modules[name]  # re-running this cell after a pull uses the new code

import html as _html
import torch
from IPython.display import HTML, Markdown, display

import yaml
CONFIG = os.environ.get("HEAT_CONFIG", "configs/heat_v3.yaml")  # the registered rule: heat_v3 = Yahoo daily, 2000 on
CFG = yaml.safe_load(open(CONFIG))
EXP, SOURCE = CFG["experiment"], CFG["data"].get("source", "sip")
BASE = "/content/drive/MyDrive/entropy-gate-pipeline-data"
DATA = os.environ.get("HEAT_DATA_DIR", f"{BASE}/data/yahoo" if SOURCE == "yahoo" else f"{BASE}/data/bars_sip")
OUT = os.environ.get("HEAT_OUT_DIR", f"{BASE}/outputs/heat_v1")  # holds the heat_v1 tuning record
REG_OUT = os.path.join(os.path.dirname(OUT), EXP)  # this experiment's results
# Each experiment has its own panel file. The 5-minute SIP panel (~420 MB) stays on this machine's disk (a huge
# Drive upload can hold up the small result files); the daily Yahoo panel is small and is kept on Drive.
PANEL = os.environ.get("HEAT_PANEL", os.path.join(REG_OUT, "panel.npz") if SOURCE == "yahoo"
                       else f"/content/heat_panel_{EXP}.npz")


def sync_drive():
    """Never unmounts Drive (an earlier version did, and then files went to this machine's disk under a
    path that only looked like Drive). Checks Drive is really mounted and asks the OS to write files out;
    Google uploads them in the background. The download cells are the sure way to keep results."""
    if not IN_COLAB:
        return
    if not os.path.ismount("/content/drive"):
        print("WARNING: Google Drive is NOT mounted, so files are only on this machine and vanish with it.")
        print("Use the 'Keep a copy' download cells, or re-run Setup to mount Drive.")
        return
    os.sync()
    print("Drive: files written; Google uploads them in the background (the download cells are the sure copy).")


def download_results(out_dir, name):
    """Zip a run's results (reports, tables, the visual report; not the model weights) and download it
    to your computer. A copy that doesn't depend on Drive."""
    import zipfile
    out_dir = os.path.abspath(out_dir)
    if not os.path.isdir(out_dir):
        print("Nothing to download yet:", out_dir, "does not exist.")
        return
    path = f"/content/{name}.zip" if IN_COLAB else os.path.join(out_dir, f"{name}.zip")
    keep = (".md", ".csv", ".json", ".html", ".yaml")
    n = 0
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(out_dir):
            for f in files:
                if f.endswith(keep):
                    z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), out_dir))
                    n += 1
    print(f"{n} files in {path} ({os.path.getsize(path) / 1e6:.1f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(path)


def show_viz(out_dir, cfg_path="configs/heat_v1.yaml", height=1800):
    """Write <out_dir>/viz.html and show it here (it is also a file you can open full-screen)."""
    from heat import config
    from heat.viz import write_report
    cfg = config.load(cfg_path)
    meta = {"experiment": cfg["experiment"], "cost_bps": cfg["evaluation"]["cost_bps"],
            "level": cfg["evaluation"]["level"], "check": os.path.basename(out_dir) == "check"}
    path = write_report(out_dir, cfg["data"]["universe"], meta, PANEL)
    print("saved", path, "- download it from Drive to open it full-screen in any browser")
    page = open(path).read()
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # IPython suggests IFrame, which can't read a Drive file
        display(HTML('<iframe srcdoc="%s" width="100%%" height="%d" style="border:1px solid #8885;border-radius:8px"></iframe>'
                     % (_html.escape(page, quote=True), height)))


ARCHS = ("bilstm", "resnet1d", "transformer")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    DEVICE = {a: "cuda" for a in ARCHS}
else:
    print("=" * 78)
    print("No NVIDIA GPU. A100 / L4 / T4 GPU runtimes are the tested, fastest path (Runtime -> Change")
    print("runtime type). On a TPU runtime, testing whether the TPU can be used instead (about a minute)...")
    print("=" * 78)
    # In a separate process that exits when done: a TPU can be held by one process at a time, and a test
    # run inside this notebook would keep it from every script started afterwards.
    import json
    _lines = !{sys.executable} scripts/pick_device.py
    print("\n".join(l for l in _lines[:-1] if l.strip()))
    DEVICE = json.loads(_lines[-1])
    if set(DEVICE.values()) == {"cpu"}:
        print("Only the CPU works here: training will be slow. Switch to a GPU runtime if you can.")
DEVICE_ALL = max(set(DEVICE.values()), key=list(DEVICE.values()).count)  # for cells that run all three
print("training on:", DEVICE)
if SOURCE == "yahoo":
    print("data: Yahoo Finance daily bars, downloaded once to", DATA,
          "| already downloaded:", any(f.startswith("yahoo_daily") for f in os.listdir(DATA)) if os.path.isdir(DATA) else False)
else:
    print("SIP bars found:", os.path.isdir(os.path.join(DATA, "5Min")))
print("registered config:", CONFIG, "->", EXP, "| results:", REG_OUT)

Mounted at /content/drive
GPU: NVIDIA L4
training on: {'bilstm': 'cuda', 'resnet1d': 'cuda', 'transformer': 'cuda'}
data: Yahoo Finance daily bars, downloaded once to /content/drive/MyDrive/entropy-gate-pipeline-data/data/yahoo | already downloaded: True
registered config: configs/heat_v3.yaml -> heat_v3 | results: /content/drive/MyDrive/entropy-gate-pipeline-data/outputs/heat_v3


## 2. Controls (optional: only after the code changed, 5–10 minutes)

Before real data: on synthetic stocks with a planted leader, the experiment must find the edge and
name the leader in the heat map; with nothing planted it must find nothing. Plus the no-look-ahead
checks, the tuning checks, the visual-report checks and the frozen-config check.

In [3]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} -m pytest -q tests -W ignore

..................................................................       [100%]
66 passed in 68.56s (0:01:08)


## 3. Check run on the real data (inside the training years)

Builds the panel once (downloads the data the first time), then a short run scored on years **inside the
registered training period**: for heat_v3, train to 2010, validate 2011–2012, score 2013–2014. It never touches
the test days, so it's safe to run as often as needed. It shows whether everything works and gives a feel for
the numbers; it is not the result.

In [4]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/run.py --config "$CONFIG" --data-dir "$DATA" --out-dir "$REG_OUT" --check --panel "$PANEL" --device "$DEVICE_ALL"
    report = os.path.join(REG_OUT, "check", "report.md")
    if os.path.exists(report):
        display(Markdown(open(report).read()))
    else:
        print("No report: the run above stopped with an error. Send a screenshot of its last lines.")
    sync_drive()  # make sure what the script wrote is really on Drive

panel loaded: 6,763 rows x 500 stocks
bilstm: already scored, skipped (pass rerun to score again)
resnet1d: already scored, skipped (pass rerun to score again)
transformer: already scored, skipped (pass rerun to score again)
# heat_v3: transfer between stocks (peers chosen by transfer entropy)

> **CHECK RUN, not the registered result.** Scored on 2013-2014 (inside the registered training years); the registered test days were not used.

Of the transfer-entropy peers, **49% are negative relations** (the stock moved against the peer) and the rest positive.

Moments: train 790,268, val 213,002, test 221,559 (scored days: after 2012-12-31). Intervals: 99% 21-day-block bootstrap. Cost: 2 bps per position.

| arch | hold | G peers (nats) | G peers − random | G TE − correlation peers | net bps (peers) | net − no peers | IC | verdict |
|---|---|---|---|---|---|---|---|---|
| bilstm | 1 d | -0.00017 [-0.00033, -0.00001] | -0.00011 [-0.00024, +0.00007] | -0.00027 [-0.00051, +0.00004] | -0.94 [-2

# heat_v3: transfer between stocks (peers chosen by transfer entropy)

> **CHECK RUN, not the registered result.** Scored on 2013-2014 (inside the registered training years); the registered test days were not used.

Of the transfer-entropy peers, **49% are negative relations** (the stock moved against the peer) and the rest positive.

Moments: train 790,268, val 213,002, test 221,559 (scored days: after 2012-12-31). Intervals: 99% 21-day-block bootstrap. Cost: 2 bps per position.

| arch | hold | G peers (nats) | G peers − random | G TE − correlation peers | net bps (peers) | net − no peers | IC | verdict |
|---|---|---|---|---|---|---|---|---|
| bilstm | 1 d | -0.00017 [-0.00033, -0.00001] | -0.00011 [-0.00024, +0.00007] | -0.00027 [-0.00051, +0.00004] | -0.94 [-2.30, +0.98] | -2.55 [-4.40, -0.19] | +0.0089 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| bilstm | 2 d | -0.00017 [-0.00043, +0.00007] | -0.00006 [-0.00030, +0.00019] | -0.00015 [-0.00064, +0.00050] | +2.20 [-1.46, +6.14] | -2.08 [-4.96, +0.70] | +0.0178 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, fdr, beats_correlation_peers) |
| bilstm | 5 d | -0.00015 [-0.00042, +0.00012] | +0.00001 [-0.00030, +0.00038] | +0.00037 [-0.00048, +0.00147] | +3.33 [-3.29, +9.07] | -4.92 [-9.15, +1.79] | +0.0129 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| bilstm | 10 d | -0.00077 [-0.00145, -0.00017] | -0.00039 [-0.00109, +0.00023] | -0.00017 [-0.00094, +0.00104] | +1.23 [-8.24, +8.93] | -13.40 [-23.17, +0.04] | +0.0025 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| resnet1d | 1 d | -0.00135 [-0.00177, -0.00100] | -0.00037 [-0.00074, -0.00000] | +0.00016 [-0.00031, +0.00063] | -2.06 [-3.13, -0.83] | -1.62 [-3.61, +0.54] | +0.0006 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| resnet1d | 2 d | -0.00269 [-0.00324, -0.00215] | -0.00132 [-0.00182, -0.00056] | -0.00172 [-0.00223, -0.00126] | -2.01 [-4.44, -0.02] | -4.92 [-7.99, -1.32] | +0.0003 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| resnet1d | 5 d | -0.00164 [-0.00283, -0.00066] | -0.00038 [-0.00176, +0.00066] | -0.00066 [-0.00214, +0.00098] | -0.35 [-4.35, +3.41] | -6.00 [-15.14, +3.42] | +0.0024 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| resnet1d | 10 d | -0.00331 [-0.00519, -0.00172] | -0.00012 [-0.00140, +0.00106] | -0.00202 [-0.00417, +0.00014] | +5.54 [-6.31, +16.41] | -11.52 [-31.30, +5.70] | +0.0068 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, fdr, beats_correlation_peers) |
| transformer | 1 d | -0.00131 [-0.00180, -0.00097] | +0.00174 [+0.00121, +0.00228] | +0.00100 [+0.00033, +0.00153] | -1.15 [-2.52, +0.13] | -0.46 [-3.06, +2.19] | +0.0043 | NO EDGE (failed: net_positive, beats_no_peers, both_halves_positive, fdr) |
| transformer | 2 d | -0.00121 [-0.00173, -0.00079] | +0.00158 [+0.00100, +0.00222] | -0.00017 [-0.00081, +0.00047] | -0.95 [-3.60, +1.92] | +0.81 [-3.47, +3.97] | +0.0045 | NO EDGE (failed: net_positive, beats_no_peers, both_halves_positive, fdr, beats_correlation_peers) |
| transformer | 5 d | -0.00313 [-0.00415, -0.00229] | -0.00150 [-0.00264, -0.00047] | -0.00170 [-0.00295, -0.00082] | -0.06 [-6.07, +6.38] | -3.77 [-15.82, +7.54] | +0.0094 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| transformer | 10 d | -0.00451 [-0.00617, -0.00318] | -0.00242 [-0.00435, -0.00098] | -0.00283 [-0.00465, -0.00166] | +3.35 [-8.05, +14.97] | -6.12 [-25.07, +12.12] | +0.0081 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |

**Cells with an edge: 0 of 12.** Cells with heat transfer (transfer-entropy peers beat random peers and no peers): 0 of 12.

## Where the heat goes

Pairs whose share of the follower's error clears zero by 3 standard errors: **112 of 17,329** (a screening bar across many pairs, not a test).

| arch | hold | source | target | moments | heat share | lower bound |
|---|---|---|---|---|---|---|
| transformer | 10 d | FSLR | CAT | 50 | +0.0467 | +0.0146 |
| resnet1d | 10 d | MAS | NTRS | 54 | +0.0399 | +0.0116 |
| resnet1d | 5 d | LHX | WRB | 50 | +0.0279 | +0.0116 |
| resnet1d | 10 d | FE | DPZ | 64 | +0.0318 | +0.0078 |
| transformer | 10 d | IDXX | TGT | 55 | +0.0131 | +0.0073 |
| resnet1d | 5 d | AIZ | BRK.B | 50 | +0.0257 | +0.0061 |
| resnet1d | 10 d | HCA | CHRW | 50 | +0.0178 | +0.0059 |
| resnet1d | 1 d | DXCM | SYY | 85 | +0.0260 | +0.0058 |
| resnet1d | 10 d | CIEN | MLM | 64 | +0.0352 | +0.0056 |
| resnet1d | 10 d | AMZN | CHTR | 52 | +0.0240 | +0.0054 |
| resnet1d | 10 d | TMUS | L | 83 | +0.0641 | +0.0051 |
| resnet1d | 10 d | PFG | GL | 81 | +0.0387 | +0.0051 |
| resnet1d | 1 d | LULU | COHR | 68 | +0.0184 | +0.0049 |
| resnet1d | 10 d | MAA | SWK | 53 | +0.0122 | +0.0034 |
| transformer | 10 d | HST | ES | 86 | +0.0099 | +0.0031 |
| resnet1d | 1 d | MCHP | IBM | 62 | +0.0139 | +0.0031 |
| resnet1d | 10 d | MLM | RL | 71 | +0.0180 | +0.0030 |
| resnet1d | 10 d | SPG | V | 66 | +0.0151 | +0.0030 |
| bilstm | 10 d | ATO | COST | 96 | +0.0063 | +0.0028 |
| transformer | 10 d | DAL | NRG | 51 | +0.0183 | +0.0027 |


Drive: files written; Google uploads them in the background (the download cells are the sure copy).


### The check run, drawn

Transfer by hold, who leads whom (matrix and network), industries leading industries, and who moves
with whom. These are the check run's numbers, not the result.

In [5]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    if os.path.exists(os.path.join(REG_OUT, "check", "cells.csv")) or os.path.exists(os.path.join(REG_OUT, "check", "cells_partial.csv")):
        show_viz(os.path.join(REG_OUT, "check"), CONFIG)
    else:
        print("No check results yet: run the check cell above first.")

saved /content/drive/MyDrive/entropy-gate-pipeline-data/outputs/heat_v3/check/viz.html - download it from Drive to open it full-screen in any browser


**Keep a copy.** Downloads the check run's results to your computer as a zip (reports, tables, the
visual report). It doesn't depend on Drive, so do it even if Drive is slow. Send the zip, or a screenshot,
if Claude can't see the files on Drive.

In [6]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    download_results(os.path.join(REG_OUT, "check"), EXP + "_check_results")

11 files in /content/heat_v3_check_results.zip (1.1 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Tuning: DONE, nothing to run here

Tuning was done once, on heat_v1's validation days (2026-10-22): for all three models no setting beat the
defaults by more than the seed-to-seed noise, so the defaults were kept. heat_v2 and heat_v3 use them; their
registered configs say so. The tuning code is still in the repo (`scripts/tune.py`) for the record, but there
is nothing to run in this notebook.

## 5. The registered run: scores the test days

One cell trains and scores all three architectures. **Nothing is computed twice:** the peer and TE tables are
cached after the first run (the check run already made them), and every trained variant is saved the moment it
finishes. If the session drops, just run the cell again: finished variants are loaded, not retrained, and
it carries on from where it stopped.

In [7]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    print("registered config:", CONFIG, "->", EXP)
    print("results go to:", REG_OUT)
    print("this scores the test days once (" + str(CFG["split"]["val_end"]) + " is the last validation day)")

registered config: configs/heat_v3.yaml -> heat_v3
results go to: /content/drive/MyDrive/entropy-gate-pipeline-data/outputs/heat_v3
this scores the test days once (2018-12-31 is the last validation day)


In [8]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/run.py --config "$CONFIG" --data-dir "$DATA" --out-dir "$REG_OUT" --panel "$PANEL" --device $DEVICE_ALL
    sync_drive()  # make sure what the script wrote is really on Drive

panel loaded: 6,763 rows x 500 stocks
peers: k=5, 756 days, monthly
peers: loaded from cache (peers_42895fcc98a60f40.npz)
moments: train 1,234,367, val 463,470, test 966,370
  bilstm/own epoch 1: val 0.63006 (7s)
  bilstm/own epoch 2: val 0.63029 (6s)
  bilstm/own epoch 3: val 0.63021 (6s)
  bilstm/own epoch 4: val 0.63038 (6s)
  bilstm/own epoch 5: val 0.63043 (6s)
  bilstm/peers epoch 1: val 0.63062 (6s)
  bilstm/peers epoch 2: val 0.63109 (6s)
  bilstm/peers epoch 3: val 0.63075 (6s)
  bilstm/peers epoch 4: val 0.63075 (7s)
  bilstm/peers epoch 5: val 0.63101 (7s)
  bilstm/random epoch 1: val 0.63040 (7s)
  bilstm/random epoch 2: val 0.63127 (7s)
  bilstm/random epoch 3: val 0.63118 (6s)
  bilstm/random epoch 4: val 0.63077 (6s)
  bilstm/random epoch 5: val 0.63166 (6s)
  bilstm/corr epoch 1: val 0.63029 (6s)
  bilstm/corr epoch 2: val 0.63110 (6s)
  bilstm/corr epoch 3: val 0.63077 (6s)
  bilstm/corr epoch 4: val 0.63066 (6s)
  bilstm/corr epoch 5: val 0.63185 (6s)
bilstm: scored a

In [9]:
download_results(REG_OUT, EXP + "_results")

22 files in /content/heat_v3_results.zip (8.5 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Result

In [10]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    report = os.path.join(REG_OUT, "report.md")
    if os.path.exists(report):
        display(Markdown(open(report).read()))
    else:
        print("No report yet: run the cells in section 5 first.")

# heat_v3: transfer between stocks (peers chosen by transfer entropy)

Of the transfer-entropy peers, **49% are negative relations** (the stock moved against the peer) and the rest positive.

Moments: train 1,234,367, val 463,470, test 966,370 (scored days: after 2018-12-31). Intervals: 99% 21-day-block bootstrap. Cost: 2 bps per position.

| arch | hold | G peers (nats) | G peers − random | G TE − correlation peers | net bps (peers) | net − no peers | IC | verdict |
|---|---|---|---|---|---|---|---|---|
| bilstm | 1 d | -0.00023 [-0.00058, +0.00002] | -0.00006 [-0.00015, +0.00002] | +0.00002 [-0.00031, +0.00041] | -0.04 [-3.26, +3.11] | -0.67 [-2.04, +0.57] | +0.0063 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| bilstm | 2 d | -0.00034 [-0.00076, -0.00005] | -0.00012 [-0.00030, +0.00003] | +0.00000 [-0.00047, +0.00058] | +1.76 [-4.66, +7.89] | -0.84 [-2.39, +0.78] | +0.0088 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, fdr, beats_correlation_peers) |
| bilstm | 5 d | -0.00068 [-0.00126, -0.00025] | -0.00036 [-0.00065, -0.00009] | +0.00014 [-0.00097, +0.00139] | +3.24 [-9.09, +14.74] | -0.85 [-4.84, +3.36] | +0.0053 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, fdr, beats_correlation_peers) |
| bilstm | 10 d | -0.00077 [-0.00115, -0.00040] | -0.00046 [-0.00080, -0.00008] | +0.00054 [-0.00041, +0.00170] | -1.61 [-16.45, +11.98] | -1.26 [-9.54, +7.53] | -0.0001 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| resnet1d | 1 d | -0.00063 [-0.00128, -0.00017] | -0.00046 [-0.00086, -0.00017] | -0.00032 [-0.00110, +0.00021] | -2.00 [-3.19, -0.93] | -2.09 [-4.57, +0.04] | -0.0005 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| resnet1d | 2 d | -0.00180 [-0.00275, -0.00118] | -0.00118 [-0.00182, -0.00067] | -0.00050 [-0.00212, +0.00055] | -1.86 [-3.75, -0.25] | -2.40 [-6.73, +1.59] | +0.0008 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| resnet1d | 5 d | -0.00136 [-0.00259, -0.00058] | -0.00020 [-0.00099, +0.00048] | +0.00350 [+0.00067, +0.00682] | -1.37 [-6.65, +3.43] | -6.34 [-14.39, +1.37] | +0.0014 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| resnet1d | 10 d | -0.00182 [-0.00356, -0.00065] | -0.00079 [-0.00214, +0.00019] | +0.00435 [+0.00128, +0.00789] | +0.87 [-9.30, +11.06] | -9.95 [-24.52, +3.70] | +0.0013 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr) |
| transformer | 1 d | -0.00072 [-0.00107, -0.00040] | -0.00030 [-0.00045, -0.00018] | -0.00013 [-0.00029, +0.00004] | -1.87 [-3.26, -0.72] | -2.19 [-5.20, +0.81] | +0.0021 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| transformer | 2 d | -0.00082 [-0.00122, -0.00050] | -0.00037 [-0.00064, -0.00019] | -0.00018 [-0.00043, +0.00009] | -2.56 [-4.69, -0.43] | -5.32 [-11.69, +0.79] | +0.0003 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| transformer | 5 d | -0.00076 [-0.00159, -0.00012] | -0.00015 [-0.00045, +0.00015] | -0.00013 [-0.00054, +0.00027] | -1.01 [-5.13, +3.44] | -5.93 [-17.55, +5.56] | +0.0012 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |
| transformer | 10 d | -0.00113 [-0.00222, -0.00013] | -0.00040 [-0.00088, +0.00006] | -0.00014 [-0.00074, +0.00052] | -1.89 [-10.00, +5.96] | -10.55 [-31.45, +10.73] | -0.0023 | NO EDGE (failed: net_positive, beats_no_peers, beats_random_peers, both_halves_positive, fdr, beats_correlation_peers) |

**Cells with an edge: 0 of 12.** Cells with heat transfer (transfer-entropy peers beat random peers and no peers): 0 of 12.

## Where the heat goes

Pairs whose share of the follower's error clears zero by 3 standard errors: **1,014 of 116,623** (a screening bar across many pairs, not a test).

| arch | hold | source | target | moments | heat share | lower bound |
|---|---|---|---|---|---|---|
| resnet1d | 10 d | PLTR | CSCO | 140 | +0.0627 | +0.0319 |
| resnet1d | 10 d | RCL | LOW | 74 | +0.0812 | +0.0196 |
| resnet1d | 10 d | RCL | VTR | 59 | +0.0685 | +0.0176 |
| resnet1d | 10 d | WYNN | MLM | 50 | +0.0486 | +0.0174 |
| resnet1d | 10 d | HRL | REG | 123 | +0.0355 | +0.0165 |
| resnet1d | 10 d | KLAC | PYPL | 55 | +0.0607 | +0.0160 |
| resnet1d | 10 d | ECHO | ORCL | 79 | +0.0353 | +0.0154 |
| resnet1d | 10 d | ALB | DHR | 57 | +0.0350 | +0.0137 |
| resnet1d | 10 d | INCY | TJX | 54 | +0.0392 | +0.0135 |
| transformer | 10 d | BE | FISV | 71 | +0.0299 | +0.0133 |
| resnet1d | 10 d | PYPL | ISRG | 64 | +0.0346 | +0.0133 |
| resnet1d | 10 d | EXPE | MRSH | 62 | +0.0393 | +0.0133 |
| resnet1d | 5 d | HAL | IEX | 66 | +0.0670 | +0.0131 |
| resnet1d | 1 d | WSM | FERG | 52 | +0.0833 | +0.0130 |
| resnet1d | 10 d | MA | FRT | 59 | +0.0350 | +0.0121 |
| resnet1d | 10 d | APP | PPG | 68 | +0.0657 | +0.0119 |
| transformer | 10 d | KLAC | PYPL | 56 | +0.0206 | +0.0114 |
| resnet1d | 10 d | CNP | ALL | 75 | +0.0364 | +0.0114 |
| resnet1d | 10 d | INTC | VTRS | 55 | +0.0436 | +0.0113 |
| resnet1d | 10 d | EMR | IR | 57 | +0.0365 | +0.0107 |


### The result, drawn

Use the filters at the top of the report: architecture (or all three averaged), hold, and how many
links to show. Hover any square, arrow or dot for the numbers. Every chart has a table view.

In [11]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    if os.path.exists(os.path.join(REG_OUT, "cells.csv")) or os.path.exists(os.path.join(REG_OUT, "cells_partial.csv")):
        show_viz(REG_OUT, CONFIG)
    else:
        print("No results yet: run the cells in section 5 first.")

saved /content/drive/MyDrive/entropy-gate-pipeline-data/outputs/heat_v3/viz.html - download it from Drive to open it full-screen in any browser


## 7. Macro release days (exploratory)

Does any transfer, or the edge, show up on **jobs report, CPI, GDP or FOMC days** rather than ordinary days?
This reuses the models already trained (their saved test predictions), so nothing is retrained; it is
exploratory, not a registered test, so it gives no verdict.

- Release dates are the **official schedules** (BLS jobs and CPI, BEA GDP, Federal Reserve FOMC), via FRED.
  Jobs, CPI and GDP come out at 8:30 am, before the open, so they are split by day. FOMC decisions come at
  2:00 pm, during trading, so FOMC is also split by whether a hold spans 2:00 pm.
- **One-time setup:** create a free FRED API key (fred.stlouisfed.org → My Account → API Keys) and add it
  as a Colab secret named `FRED_API_KEY` (key icon on the left, notebook access on). The calendar is saved
  the first time, so later runs don't need the key.

**7a. On the check run:**

In [12]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/events.py --config "$CONFIG" --data-dir "$DATA" --out-dir "$REG_OUT" --panel "$PANEL" --check
    sync_drive()  # make sure what the script wrote is really on Drive

panel loaded: 6,763 rows x 500 stocks
Traceback (most recent call last):
  File "/content/Transfer-entropy-/scripts/events.py", line 51, in <module>
    main()
    ~~~~^^
  File "/content/Transfer-entropy-/scripts/events.py", line 43, in main
    calendar = load_or_fetch(args.calendar or str(out / "events_calendar.csv"))
  File "/content/Transfer-entropy-/heat/events.py", line 75, in load_or_fetch
    cal = fetch_calendar(start, end)
  File "/content/Transfer-entropy-/heat/events.py", line 53, in fetch_calendar
    key = fred_key(api_key)
  File "/content/Transfer-entropy-/heat/events.py", line 46, in fred_key
    raise RuntimeError("FRED needs a free API key: create one at fred.stlouisfed.org (My Account -> API Keys), "
                       "then add it as the environment variable or Colab secret FRED_API_KEY")
RuntimeError: FRED needs a free API key: create one at fred.stlouisfed.org (My Account -> API Keys), then add it as the environment variable or Colab secret FRED_API_KEY
Driv

**7b. On the registered result (the test days), after section 5:**

In [13]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/events.py --panel "$PANEL" --config "$CONFIG" --data-dir "$DATA" --out-dir $REG_OUT
    sync_drive()  # make sure what the script wrote is really on Drive

panel loaded: 6,763 rows x 500 stocks
Traceback (most recent call last):
  File "/content/Transfer-entropy-/scripts/events.py", line 51, in <module>
    main()
    ~~~~^^
  File "/content/Transfer-entropy-/scripts/events.py", line 43, in main
    calendar = load_or_fetch(args.calendar or str(out / "events_calendar.csv"))
  File "/content/Transfer-entropy-/heat/events.py", line 75, in load_or_fetch
    cal = fetch_calendar(start, end)
  File "/content/Transfer-entropy-/heat/events.py", line 53, in fetch_calendar
    key = fred_key(api_key)
  File "/content/Transfer-entropy-/heat/events.py", line 46, in fred_key
    raise RuntimeError("FRED needs a free API key: create one at fred.stlouisfed.org (My Account -> API Keys), "
                       "then add it as the environment variable or Colab secret FRED_API_KEY")
RuntimeError: FRED needs a free API key: create one at fred.stlouisfed.org (My Account -> API Keys), then add it as the environment variable or Colab secret FRED_API_KEY
Driv

**Keep a copy** of the registered result on your computer (independent of Drive).

In [14]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    download_results(REG_OUT, EXP + "_results")

22 files in /content/heat_v3_results.zip (8.5 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>